# 7장 — 과적합: 통째로 외운 학생 📚 (세 번째 핵심)

재단사의 유혹(차수 스위프)과 벌어짐 감시자(훈련 곡선)를 재현합니다.

**이 노트북의 목표**
1. 차수(용량)를 올리면 훈련 오차는 0으로, 검증 오차는 U자. 훈련 오차 0 = 암기
2. 데이터를 4배로 늘리면 같은 용량이 과적합하지 못한다 (용량 대 데이터의 힘겨루기)
3. GD 훈련 곡선의 벌어짐: 검증 최저점이 멈출 자리

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch7')

## 1. 실험 ① · 재단사의 유혹: 차수 스위프

숨은 규칙 sin(2.5x)에서 노이즈 낀 점 10개가 관측됐습니다. 차수 0~9 다항식을 맞추고,
봉인해 둔 검증 30점으로 진짜 시험을 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

TRUE = lambda x: np.sin(2.5 * x)   # 숨은 규칙
SIGMA = 0.25                        # 관측 노이즈

def make_data(n, seed):
    rng = np.random.default_rng(seed)
    x = np.clip(np.linspace(-1, 1, n) + rng.uniform(-0.04, 0.04, n), -1, 1)
    return x, TRUE(x) + SIGMA * rng.standard_normal(n)

xt, yt = make_data(10, 21)   # 훈련 (연습 문제)
xv, yv = make_data(30, 22)   # 검증 (봉인된 시험)

degs = range(10)
trs, vas = [], []
for d in degs:
    c = np.polyfit(xt, yt, d)
    trs.append(np.mean((np.polyval(c, xt) - yt) ** 2))
    vas.append(np.mean((np.polyval(c, xv) - yv) ** 2))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
gx = np.linspace(-1, 1, 200)
axes[0].plot(gx, TRUE(gx), 'k--', alpha=0.4, label='숨은 규칙')
for d, c_ in zip([1, 3, 9], ['tab:blue', 'tab:green', 'tab:red']):
    w = np.polyfit(xt, yt, d)
    axes[0].plot(gx, np.clip(np.polyval(w, gx), -2.5, 2.5), color=c_, label=f'차수 {d}')
axes[0].scatter(xt, yt, c='indigo', zorder=5, s=30, label='훈련 점 10개')
axes[0].set_ylim(-2.5, 2.5); axes[0].legend(fontsize=8); axes[0].set_title('맞춘 곡선들')
axes[1].semilogy(degs, trs, 'o-', label='훈련 오차')
axes[1].semilogy(degs, vas, 's-', color='tab:red', label='검증 오차')
axes[1].set_xlabel('차수(용량)'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title('훈련은 0으로, 검증은 U자로')
plt.tight_layout(); plt.show()

for d in degs:
    print(f'  차수 {d}: 훈련 {trs[d]:.4f}  검증 {vas[d]:.4f}')

best_mid = min(vas[3:8])
assert trs[9] < 1e-6, '차수 9 = 다이얼 10개 ≥ 점 10개 → 훈련 오차 사실상 0 (완벽 암기)'
assert vas[9] > 3 * best_mid, '암기한 곡선의 검증 오차는 적정 용량의 3배 이상'
assert best_mid < vas[0] and best_mid < vas[9], '검증 오차는 U자 — 최적은 양 극단 사이'
print('\n✔ 확인: 훈련 오차 0은 실력이 아니라 산수. 검증(처음 보는 문제)이 진짜 점수')

## 2. 실험 ①-보너스: 데이터 4배의 힘

같은 차수 9(다이얼 10개)를 점 40개 앞에 세워 봅니다. 전부 외울 수 있을까요?

In [ ]:
xt4, yt4 = make_data(40, 23)   # 훈련 점 4배
c9 = np.polyfit(xt4, yt4, 9)
tr94 = np.mean((np.polyval(c9, xt4) - yt4) ** 2)
va94 = np.mean((np.polyval(c9, xv) - yv) ** 2)
print(f'차수 9, 점 10개: 훈련 {trs[9]:.6f}  검증 {vas[9]:.4f}   ← 완벽 암기, 시험 폭망')
print(f'차수 9, 점 40개: 훈련 {tr94:.4f}  검증 {va94:.4f}   ← 암기 불가, 규칙 학습')

assert tr94 > 0.02, '다이얼 10개로 40개 점은 못 외운다 — 훈련 오차가 0이 못 됨'
assert va94 < vas[9] / 2, '암기가 막히니 규칙을 배워 검증 오차가 절반 이하로'
print('\n✔ 확인: 과적합은 용량 대 데이터의 힘겨루기. 데이터 늘리기가 가장 정직한 처방')

## 3. 실험 ② · 벌어짐 감시자: 훈련 일지의 무늬

실전에선 차수 슬라이더가 아니라 훈련 스텝이 흐릅니다. 차수 9(암기 가능 용량)를
경사하강으로 3,000스텝 돌리며 두 손실을 함께 기록합니다.
(체비쇼프 기저. 거듭제곱 기저는 조건수가 나빠 GD가 겉돕니다)

In [ ]:
def cheb(x, deg=9):
    T = [np.ones_like(x), x]
    for k in range(2, deg + 1):
        T.append(2 * x * T[-1] - T[-2])
    return np.stack(T[: deg + 1], axis=1)

Ft, Fv = cheb(xt), cheb(xv)
w = np.zeros(10)
tr_curve, va_curve = [], []
for t in range(3000):
    r = Ft @ w - yt
    w -= 0.05 * 2 * Ft.T @ r / len(yt)
    tr_curve.append(np.mean(r ** 2))
    va_curve.append(np.mean((Fv @ w - yv) ** 2))
tr_curve, va_curve = np.array(tr_curve), np.array(va_curve)
k = int(va_curve.argmin())

plt.figure(figsize=(7.5, 3.4))
plt.plot(tr_curve, label='훈련 손실', color='indigo')
plt.plot(va_curve, label='검증 손실', color='tab:red')
plt.axvline(k, color='green', ls='--', alpha=0.7, label=f'검증 최저 (스텝 {k}): 멈출 자리')
plt.ylim(0, 1.0); plt.xlabel('스텝'); plt.legend(); plt.grid(alpha=0.3)
plt.title('벌어짐: 훈련은 계속 내려가고, 검증은 돌아선다')
plt.tight_layout(); plt.show()

print(f'검증 최저 {va_curve[k]:.4f} @ 스텝 {k}')
print(f'마지막(3000스텝): 훈련 {tr_curve[-1]:.5f}  검증 {va_curve[-1]:.4f} (최저의 {va_curve[-1]/va_curve[k]:.1f}배)')

assert k < 500, '검증 최저는 훈련 초반 — 큰 패턴은 일찍 배워진다'
assert tr_curve[-1] < 1e-3, '훈련 손실은 끝까지 계속 하강 (암기 진행)'
assert va_curve[-1] > 2 * va_curve[k], '최저점 이후 검증은 2배 이상 악화 — 벌어짐'
print('\n✔ 확인: 경사하강은 규칙 먼저, 암기는 나중. 두 곡선이 벌어지는 순간이 과적합의 시작')

## 4. 관찰 정리 + 🤖 AI 연결

- **오차는 두 개**: 훈련 오차(연습 점수)와 검증 오차(처음 보는 문제). 훈련 데이터는
  표본(3부)이므로, 성적은 표본 밖에서 매겨진다(**일반화**). 일반화 갭 = 검증 − 훈련.
- **훈련 오차 0의 정체**: 다이얼 수 ≥ 데이터 수면 완벽 암기가 항상 가능. 실력이 아니라 산수.
  검증 오차는 U자(과소적합 ↔ 과적합). 데이터를 늘리면 암기가 막혀 규칙을 배울 수밖에.
- **벌어짐 무늬**: GD는 큰 패턴 먼저, 노이즈 암기는 나중 → 검증 손실은 내려가다 돌아선다.
  두 곡선이 벌어지는 지점 = 과적합 시작, 검증 최저 = 멈출 자리(조기 종료, 8장).
- 실전(6부 9장 복습): SFT 소량 데이터 × 여러 에포크에서 `eval_loss` 상승 = 예문 암기.
  에포크 줄이기·데이터 늘리기, 그리고 다음 장의 안전장치.

**다음 챕터**: 8장, 안전장치들(정규화). weight decay(원점 목줄)·드롭아웃·조기 종료.
6부 6장 KL 목줄("멀리 못 가게 묶는다")의 형제들입니다.